In [1]:
import logging
import pickle
import time

import matplotlib.pyplot as plt
import nest
import numpy as np
import sudoku_net
from helpers_sudoku import get_puzzle, plot_field, validate_solution

import neuroring_sudoku
from utils_binding import *   # provides .index and .bitstreamFile
import pyxrt
import re


nest.SetKernelStatus({"local_num_threads": 8})
nest.set_verbosity("M_WARNING")
logging.basicConfig(level=logging.INFO)

puzzle_index = 8
noise_rate = 200
sim_time = 100
stim_rate = 200
max_sim_time = 10000
max_iterations = max_sim_time // sim_time

puzzle = get_puzzle(puzzle_index)
network = sudoku_net.SudokuNet(pop_size=5, input=puzzle, noise_rate=noise_rate, stim_rate=stim_rate)

solution_states = np.zeros((max_iterations, 9, 9), dtype=np.int_)

param_dict = {
    'dt': 0.1,
    'tau_m': 20.0,
    'tau_syn': 5.0,
    'C_m': 250.0,
    'E_L': -65.0,
    't_ref_steps': 20,
    'V_th_abs': -50.0,
    'V_reset_abs': -70.0,
}
record_status = 1
host = neuroring_sudoku.NeuroRingHost(network, 4096, 7000, 1, 1, param_dict, record_status, "/home/miahafiz/NeuroRing/_build_dir.hw.NUM_4096.CORE_10.FREQ_300/krnl_neuroring_hw.xclbin")
host.initialize_devices()
print("Initialized devices")

import subprocess

POWER_DEVICES = ["0000:2a:00.1"]  # only one U55C device

def measure_board_power(device):
    """Return power (W) for a single board using xrt-smi, or None on failure."""
    try:
        out = subprocess.check_output(
            ["xrt-smi", "examine", "-d", device, "-r", "electrical"],
            text=True
        )
    except Exception as e:
        print(f"Failed to read power for {device}: {e}")
        return None

    m = re.search(r"^\s*Power\s+:\s*([\d.]+)\s*Watts", out, re.MULTILINE)
    if not m:
        print(f"Could not parse power for {device}")
        return None
    return float(m.group(1))

def measure_total_power():
    """Sum power over all devices in POWER_DEVICES."""
    readings = [measure_board_power(dev) for dev in POWER_DEVICES]
    readings = [p for p in readings if p is not None]
    return sum(readings) if readings else None


def reset_spike():
    host.kernels_per_fpga[0][0].upload_synapse_list(host.synapse_fpga[0])
    host.kernels_per_fpga[0][1].upload_synapse_list(host.noise_stim_fpga)

def get_power(sim_time_start, sim_time_end):
    start_time = time.time()
    sim_time = sim_time_end - sim_time_start
    host.kernels_per_fpga[0][0].run_neuroring(sim_time_start, sim_time_end)
    host.kernels_per_fpga[0][1].run_poisson(sim_time)

    host.kernels_per_fpga[0][0].run_synapserouter(sim_time)
    host.kernels_per_fpga[0][1].run_synapserouter(sim_time)
    
    time.sleep(0.01)
    power = measure_total_power()

    host.kernels_per_fpga[0][0].wait_for_kernel()
    host.kernels_per_fpga[0][1].wait_for_kernel()
    end_time = time.time()
    return power

def simulate(sim_time_start, sim_time_end):
    start_time = time.time()
    sim_time = sim_time_end - sim_time_start
    host.kernels_per_fpga[0][0].run_neuroring(sim_time_start, sim_time_end)
    host.kernels_per_fpga[0][1].run_poisson(sim_time)

    host.kernels_per_fpga[0][0].run_synapserouter(sim_time)
    host.kernels_per_fpga[0][1].run_synapserouter(sim_time)
    
    host.kernels_per_fpga[0][0].wait_for_kernel()
    host.kernels_per_fpga[0][1].wait_for_kernel()
    end_time = time.time()
    return (end_time - start_time)

def get_spike_poisson(sim_time):
    spikeidx, neuronidx = host.get_spike_recorder_array(sim_time)
    # Pair and sort based on neuronidx (low to high)
    paired = sorted(zip(spikeidx, neuronidx), key=lambda x: x[1])
    # Filter only pairs where neuronidx >= 3646
    paired = [p for p in paired if p[1] >= 3646]
    return paired

def get_spike(sim_time, n_neurons=3645, group_size=5):
    spikeidx, neuronidx = host.get_spike_recorder_array(sim_time)

    spikeidx = np.asarray(spikeidx)
    neuronidx = np.asarray(neuronidx)

    # Keep only the Sudoku population (neurons 1..n_neurons)
    mask = (neuronidx >= 1) & (neuronidx <= n_neurons)
    spikeidx = spikeidx[mask]
    neuronidx = neuronidx[mask]

    n_groups = (n_neurons + group_size - 1) // group_size
    senders = [[] for _ in range(n_groups)]
    times = [[] for _ in range(n_groups)]

    # Group by blocks of `group_size` neurons: (1..5), (6..10), ...
    for t, n in zip(spikeidx, neuronidx):
        gi = (int(n) - 1) // group_size
        if 0 <= gi < n_groups:
            senders[gi].append(int(n))
            times[gi].append(float(t))

    # Match the common `spiketrains` shape: list (len=729) of [ {'senders','times'} ]
    spiketrains = []
    for i in range(n_groups):
        if times[i]:
            order = np.argsort(times[i])
            s = np.asarray([senders[i][j] for j in order], dtype=np.int32)
            tt = np.asarray([times[i][j] for j in order], dtype=np.float32)
        else:
            s = np.asarray([], dtype=np.int32)
            tt = np.asarray([], dtype=np.float32)
        spiketrains.append([{'senders': s, 'times': tt}])

    return spiketrains


def plot(filename="spike_recorder_sudoku.png", start_tick=0, end_tick=100):
    host.plot_spike_recorder_array(filename, start_tick, end_tick)
    np.savetxt("spikeidx_sudoku.csv", host.spikeidx, delimiter=",")
    np.savetxt("neuronidx_sudoku.csv", host.neuronidx, delimiter=",")




              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0
 Built: Oct  2 2025 06:57:01

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



INFO:root:Creating neuron populations...
INFO:root:Setting up noise...
INFO:root:Creating inter-neuron and IO-connections...
INFO:root:setting input...
INFO:root:Setup complete.


Extracting synapse information...

Distributing 3645 neurons across 1 compute units on 1 FPGAs:
Kernels per FPGA: [1]

FPGA 0 (Kernels: 1):
  Kernel 0 (Global ID: 0): neurons 1 to 3645 (total: 3645)

Total neurons assigned: 3645
Initialized device 0 with XCLBIN UUID: 80f4e378-8ea1-83e7-af51-670d32726316
{'simulation_time': 1, 'amount_of_cores': 2, 'neuron_start': 1, 'neuron_total': 3645, 'core_id': 0, 'neuron_per_cu': 4096, 'synapse_total_per_cu': 7000, 'record_status': 1, 'param_dict': {'dt': 0.1, 'tau_m': 20.0, 'tau_syn': 5.0, 'C_m': 250.0, 'E_L': -65.0, 't_ref_steps': 20, 'V_th_abs': -50.0, 'V_reset_abs': -70.0}, 'device': None, 'xclbin': None, 'uuid': None, 'kernel_name': None, 'kernel': None, 'synapseListHandle': None, 'header_words': 12800000, 'header_bytes': 51200000, 'tail_words_capacity': 57344000, 'tail_bytes_capacity': 229376000, 'bo_size': 280576000}
Initialized kernel NeuroRing:{NeuroRing_0} and SynapseRouter:{SynapseRouter_0} on device <pyxrt.device object at 0x7fbe61d5f9

In [3]:
print(len(host.synapse_data))
offset = 7000
print(host.synapse_data[0*offset][0])
print(host.synapse_data[1*offset][0])

accumulated_synapse = 0
for i in range(host.total_neurons):
    accumulated_synapse += host.synapse_data[i*offset][0]
print(accumulated_synapse)

average_synapse = accumulated_synapse / host.total_neurons
print(average_synapse)

25515000
141.0
141.0
513945.0
141.0


In [2]:
print(puzzle)

[[0 5 8 0 3 0 0 2 0]
 [4 0 2 0 0 0 9 0 5]
 [0 0 7 0 0 0 6 8 0]
 [2 9 0 0 5 4 0 7 0]
 [5 0 0 0 6 2 0 0 0]
 [0 0 3 8 1 0 2 5 0]
 [1 0 9 0 0 3 0 6 4]
 [8 6 5 4 9 0 1 3 0]
 [0 7 0 0 0 6 0 0 0]]


In [3]:
simulation_time = 10000
reset_spike()
exec_time = simulate(0,simulation_time)
print(f"time execution: {exec_time}")

paired_spike = get_spike(simulation_time)
plot(filename="spike_recorder_sudoku.png", start_tick=0, end_tick=simulation_time)

solution = np.array(puzzle, dtype=np.uint8)
spike_recorders = network.io_indices[0, 2]
idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
cell_spikes = [paired_spike[int(i)] for i in idx]
spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])
print(spike_counts)
counts_desc = np.unique(spike_counts)[::-1]
print(counts_desc)

winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
solution[0, 0] = winning_digit

print(winning_digit)


time execution: 0.1309797763824463
[  0   0   0   0   0   0   0 462   0]
[462   0]
8


In [ ]:
sim_time_start = 0
sim_time_end = 60000
reset_spike()
exec_time = simulate(sim_time_start, sim_time_end)
#print(f"time execution: {exec_time}", end="\r", flush=True)
paired_spike = get_spike(sim_time_end)

# Initialize solution as a copy of the puzzle to keep track of placed digits
solution = puzzle.copy()
solution_states = {} # Assuming this is a dict or array initialized earlier
run = 0

for row in range(9):
    for col in range(9):
        # 1. Use the given puzzle value if it's not empty (0)
        if puzzle[row, col] != 0:
            solution[row, col] = puzzle[row, col]
            continue
            
        # obtain indices of the spike recorders coding for digits in the current cell
        spike_recorders = network.io_indices[row, col]
        
        # spiketrains for all digits in the current cells
        idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
        cell_spikes = [paired_spike[int(i)] for i in idx]
        spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])
        
        # 2. Get unique spike counts in descending order (highest spikes first)
        unique_counts = np.sort(np.unique(spike_counts))[::-1]
        
        winning_digit = 0
        
        # 3. Iterate through possible spike count levels
        for count in unique_counts:
            # Get all digits (1-9) that share this specific spike count
            # Note: +1 because indices are 0-8, but Sudoku digits are 1-9
            candidates = np.flatnonzero(spike_counts == count) + 1
            
            valid_candidates = []
            
            # 4. Check Sudoku constraints for each candidate
            for digit in candidates:
                # Check row
                if digit in solution[row, :]:
                    continue
                # Check column
                if digit in solution[:, col]:
                    continue
                # Check 3x3 square
                sq_r, sq_c = (row // 3) * 3, (col // 3) * 3
                if digit in solution[sq_r:sq_r+3, sq_c:sq_c+3]:
                    continue
                    
                # If it passes all checks, it's a valid candidate
                valid_candidates.append(digit)
                
            # 5. If we have valid candidates at this spike level, pick one and stop looking
            if valid_candidates:
                # If there are multiple valid candidates with the same count, this picks randomly
                winning_digit = int(np.random.choice(valid_candidates))
                break
                
        # 6. Fallback mechanism
        # In the rare event SNN misfires so badly that ALL 9 digits violate rules 
        # (usually due to a mistake in a previously placed cell), we fall back 
        # to the absolute maximum spiking digit to at least place *something*.
        if winning_digit == 0:
            winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
            
        solution[row, col] = winning_digit

# Save and validate
solution_states[run] = solution
valid, cells, rows, cols = validate_solution(puzzle, solution)
ratio_correct = (np.sum(cells) + np.sum(rows) + np.sum(cols)) / 27

print(f"performance: {np.round(ratio_correct, 3)} cell:{np.sum(cells)} rows:{np.sum(rows)} cols:{np.sum(cols)}")
print(solution)
print("--------------------------------")
print(f"valid: {valid}")
print(f"cells: {cells}")
print(f"rows: {rows}")
print(f"cols: {cols}")


In [9]:
sim_time_start = 0
sim_time_end = 5000
start_time = time.time()
reset_spike()
exec_time = simulate(sim_time_start, sim_time_end)
print(f"simulation execution: {exec_time}")
paired_spike = get_spike(sim_time_end)

# Initialize solution as a copy of the puzzle to keep track of placed digits
solution = puzzle.copy()
solution_states = {} # Assuming this is a dict or array initialized earlier
run = 0

for row in range(9):
    for col in range(9):
        # obtain indices of the spike recorders coding for digits in
        # the current cell
        spike_recorders = network.io_indices[row, col]
        # spiketrains for all digits in the current cells
        idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
        cell_spikes = [paired_spike[int(i)] for i in idx]
        spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])
        #print(spike_counts)
        # if two digits have the same activation, pick one at random
        winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
        #print(winning_digit)
        #print("-------------------------------------------------------------------------------")
        solution[row, col] = winning_digit

# Save and validate
solution_states[run] = solution
valid, cells, rows, cols = validate_solution(puzzle, solution)
end_time = time.time()
ratio_correct = (np.sum(cells) + np.sum(rows) + np.sum(cols)) / 27

print(f"time execution: {end_time - start_time}")
print(f"performance: {np.round(ratio_correct, 3)} cell:{np.sum(cells)} rows:{np.sum(rows)} cols:{np.sum(cols)}")
print(puzzle)
print("--------------------------------")
print(solution)
print("--------------------------------")
print(f"valid: {valid}")
print(f"cells: {cells}")
print(f"rows: {rows}")
print(f"cols: {cols}")

# check non-zero digits in puzzle against the spike-based solution
mismatches = []
for row in range(9):
    for col in range(9):
        if puzzle[row, col] != 0 and solution[row, col] != puzzle[row, col]:
            mismatches.append((row, col, int(puzzle[row, col]), int(solution[row, col])))

if len(mismatches) == 0:
    print("All given puzzle digits are identical to the solution.")
else:
    print("Given-digit mismatches:")
    for row, col, p_val, s_val in mismatches:
        print(f"row={row}, col={col}, puzzle={p_val}, solution={s_val}")

simulation execution: 0.06468510627746582
time execution: 0.44536638259887695
performance: 1.0 cell:9 rows:9 cols:9
[[0 5 8 0 3 0 0 2 0]
 [4 0 2 0 0 0 9 0 5]
 [0 0 7 0 0 0 6 8 0]
 [2 9 0 0 5 4 0 7 0]
 [5 0 0 0 6 2 0 0 0]
 [0 0 3 8 1 0 2 5 0]
 [1 0 9 0 0 3 0 6 4]
 [8 6 5 4 9 0 1 3 0]
 [0 7 0 0 0 6 0 0 0]]
--------------------------------
[[6 5 8 9 3 1 4 2 7]
 [4 3 2 6 7 8 9 1 5]
 [9 1 7 2 4 5 6 8 3]
 [2 9 6 3 5 4 8 7 1]
 [5 8 1 7 6 2 3 4 9]
 [7 4 3 8 1 9 2 5 6]
 [1 2 9 5 8 3 7 6 4]
 [8 6 5 4 9 7 1 3 2]
 [3 7 4 1 2 6 5 9 8]]
--------------------------------
valid: True
cells: [[ True  True  True]
 [ True  True  True]
 [ True  True  True]]
rows: [ True  True  True  True  True  True  True  True  True]
cols: [ True  True  True  True  True  True  True  True  True]
All given puzzle digits are identical to the solution.


In [10]:
print(len(host.neuronidx))

87647


In [14]:
power = get_power(0, 5000)
print(f"power: {power}")

power: 20.789248


In [ ]:
simulation_time = 50000
reset_spike()
exec_time = simulate(0,simulation_time)
print(f"time execution: {exec_time}")

paired_spike = get_spike(simulation_time)
#plot(filename="spike_recorder_sudoku.png", start_tick=0, end_tick=simulation_time)

#solution = np.array(puzzle, dtype=np.uint8)
row0 = 7
col0 = 0
print(f"row0: {row0}, col0: {col0}")
spike_recorders = network.io_indices[row0, col0]
idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
cell_spikes = [paired_spike[int(i)] for i in idx]
spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])
print(spike_counts)
winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
print(winning_digit)
print("--------------------------------")
row0 = 8
col0 = 0
print(f"row0: {row0}, col0: {col0}")
spike_recorders = network.io_indices[row0, col0]
idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
cell_spikes = [paired_spike[int(i)] for i in idx]
spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])
print(spike_counts)
winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
print(winning_digit)

In [ ]:
run = 0
valid = False
max_iterations = 50
sim_time_start = 0
sim_time_end = 1000
reset_spike()
while not valid:
    
    exec_time = simulate(sim_time_start, sim_time_end)
    #print(f"time execution: {exec_time}", end="\r", flush=True)

    paired_spike = get_spike(sim_time_end)
    solution = np.zeros((9, 9), dtype=np.uint8)

    for row in range(9):
        for col in range(9):
            # obtain indices of the spike recorders coding for digits in
            # the current cell
            spike_recorders = network.io_indices[row, col]

            # spiketrains for all digits in the current cells
            idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
            cell_spikes = [paired_spike[int(i)] for i in idx]
            spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])

            # if two digits have the same activation, pick one at random
            winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
            solution[row, col] = winning_digit

    #solution_states[run] = solution
    valid, cells, rows, cols = validate_solution(puzzle, solution)
    #print(solution, flush=True)
    #print(cols, flush=True)
    #print(rows, flush=True)
    #print(cells, flush=True)
    #print(valid, flush=True)

    if not valid:
        ratio_correct = (np.sum(cells) + np.sum(rows) + np.sum(cols)) / 27
        print(f"{sim_time_end}ms, performance: " f"{np.round(ratio_correct, 3)}" f"cell:{np.sum(cells)} rows:{np.sum(rows)} cols:{np.sum(cols)}")
    else:
        print(f"{sim_time_end}ms, valid solution found.")
        break

    run += 1
    sim_time_start = sim_time_end
    sim_time_end += 1000
    if run >= max_iterations:
        #print(f"no solution found after {sim_time_end}ms, aborting.")
        break


In [ ]:
dt = 0.1
tau_m = 20.0
tau_syn = 5.0
C_m = 250.0
E_L = -65.0
V_decay = np.exp(-dt/tau_m)   # exp(-dt/tau_m)
I_decay = np.exp(-dt/tau_syn)   # exp(-dt/tau_syn)
syn_to_vm = (1.0 / C_m) * ((I_decay - V_decay) / ((1.0 / tau_m) - (1.0 / tau_syn)))
bias_to_vm = (tau_m / C_m) * (1.0 - V_decay)  # mV per pA
t_ref_steps = 20        # round(2.0/0.1)
V_th_abs = -50.0
V_th_rel = V_th_abs - E_L
V_reset_abs = -70.0
V_reset_rel = V_reset_abs - E_L
E_L_dec = 65.0

print(f"V_decay: {V_decay}")
print(f"I_decay: {I_decay}")
print(f"syn_to_vm: {syn_to_vm}")
print(f"bias_to_vm: {bias_to_vm}")


In [ ]:
network.reset_spike_recorders()
nest.Simulate(sim_time)
spiketrains = network.get_spike_trains()
solution = np.zeros((9, 9), dtype=np.uint8)


In [ ]:
spike_recorders = network.io_indices[0, 2]
idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
cell_spikes = [spiketrains[int(i)] for i in idx]
spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])

winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
solution[0, 2] = winning_digit

In [ ]:
print(spike_recorders)
print(cell_spikes)
print(spike_counts)
print(winning_digit)

In [ ]:
    run =0
    for row in range(9):
        for col in range(9):
            # obtain indices of the spike recorders coding for digits in
            # the current cell
            spike_recorders = network.io_indices[row, col]

            # spiketrains for all digits in the current cells
            idx = np.asarray(spike_recorders, dtype=np.int64).ravel()
            cell_spikes = [spiketrains[int(i)] for i in idx]
            spike_counts = np.array([len(s[0]["times"]) for s in cell_spikes])

            # if two digits have the same activation, pick one at random
            winning_digit = int(np.random.choice(np.flatnonzero(spike_counts == spike_counts.max()))) + 1
            solution[row, col] = winning_digit

    solution_states[run] = solution
    valid, cells, rows, cols = validate_solution(puzzle, solution)


In [ ]:
print(solution_states[0])
print(cols)

In [ ]:
def xorshift32_next(s):
    """
    Xorshift32 triplet matching NeuroRing_Poisson.cpp exactly.
    Use int(s) so numpy.uint32 doesn't change semantics; mask after each step
    so Python's arbitrary-width ints behave like C++ 32-bit.
    """
    s = int(s) & 0xFFFFFFFF
    s = (s ^ (s << 13)) & 0xFFFFFFFF
    s = (s ^ (s >> 17)) & 0xFFFFFFFF
    s = (s ^ (s << 5)) & 0xFFFFFFFF
    return s

seed_num = np.random.randint(1, 2**32, dtype=np.uint32)
poisson_prob = 1 - np.exp(-0.1 * 200)
poisson_prob_q32 = int(poisson_prob * 2**32) & 0xFFFFFFFF

random_num = xorshift32_next(seed_num)
seed_num = random_num
spike = (random_num < poisson_prob_q32)

print(seed_num)
print(random_num)
print(poisson_prob_q32)
print(spike)

import os
import struct

raw = os.urandom(4)
seed = struct.unpack('<I', raw)[0]

print(seed)
print(raw)

